In [52]:
# ============================================================================
# CELL 1: SET YOUR API KEYS HERE using Colab Secrets
# ============================================================================
import os
from google.colab import userdata

# Retrieve API keys from Colab Secrets
# Add GOOGLE_API_KEY, NGROK_AUTHTOKEN, and GROQ_API_KEY as secrets.
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
os.environ['NGROK_AUTHTOKEN'] = userdata.get('NGROK_AUTHTOKEN')
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

print("✅ Environment variables set from Colab Secrets!")
print("Google Key:", os.environ.get('GOOGLE_API_KEY')[:4] + "..." if os.environ.get('GOOGLE_API_KEY') else "NOT SET")
print("Ngrok Token:", os.environ.get('NGROK_AUTHTOKEN')[:4] + "..." if os.environ.get('NGROK_AUTHTOKEN') else "NOT SET")
print("Groq Key:", os.environ.get('GROQ_API_KEY')[:4] + "..." if os.environ.get('GROQ_API_KEY') else "NOT SET")

✅ Environment variables set from Colab Secrets!
Google Key: AIza...
Ngrok Token: 33xq...
Groq Key: gsk_...


In [58]:
# ============================================================================
# CELL 2: CLEANUP OLD TUNNELS
# ============================================================================
print("\n--- Cleaning up old ngrok tunnels... ---")
from pyngrok import ngrok
import os

ngrok.set_auth_token(os.environ.get('NGROK_AUTHTOKEN'))

try:
    tunnels = ngrok.get_tunnels()
    print(f"Found {len(tunnels)} active tunnels. Disconnecting...")
    for tunnel in tunnels:
        print(f"  Closing: {tunnel.public_url}")
        ngrok.disconnect(tunnel.public_url)
    print("✅ All old tunnels closed!")
except Exception as e:
    print(f"⚠️ Error during cleanup: {e}")

!pkill -f streamlit
print("✅ Streamlit processes killed!")


# ============================================================================
# CELL 3: INSTALL, CREATE APP, AND LAUNCH
# ============================================================================

print("\n--- STEP 1 of 3: Installing necessary packages... ---")
!pip install -q streamlit pyngrok groq
print("✅ Packages installed successfully.\n")

print("--- STEP 2 of 3: Defining and writing the Streamlit app code... ---")

APP_CODE = """
import streamlit as st
from groq import Groq
import os

st.set_page_config(layout="wide", page_title="AI Study Buddy", page_icon="📚")

# Custom CSS
st.markdown(\"\"\"
<style>
    body, .stApp {
        background-color: #121212;
        color: #e0e0e0;
        font-family: 'Segoe UI', sans-serif;
    }
    .main .block-container {
        padding-top: 2rem;
        padding-bottom: 2rem;
        background-color: #1e1e1e;
        border-radius: 12px;
        box-shadow: 0 0 20px rgba(0, 0, 0, 0.5);
    }
    .main-header {
        text-align: center;
        padding: 2rem 1rem;
        background-color: #1f2937;
        color: #f8fafc;
        border-radius: 10px;
        margin-bottom: 2rem;
    }
    h1, h2, h3, h4, h5, h6 {
        color: #f8fafc !important;
        font-weight: 600;
    }
    p, label, span {
        color: #e0e0e0 !important;
    }
    .stTextInput>div>div>input,
    .stTextArea>div>div>textarea {
        background-color: #1f2937 !important;
        color: #f8fafc !important;
        border: 1px solid #374151 !important;
        border-radius: 6px;
        font-size: 1rem;
    }
    .stSelectbox>div>div>select {
        background-color: #1f2937 !important;
        color: #f8fafc !important;
        border: 1px solid #374151 !important;
        border-radius: 6px;
    }
    .stButton>button {
        background-color: #3b82f6;
        color: white !important;
        border: none;
        padding: 0.6rem 1rem;
        font-size: 1rem;
        border-radius: 8px;
        font-weight: 600;
        transition: all 0.2s ease-in-out;
    }
    .stButton>button:hover {
        background-color: #2563eb;
        transform: translateY(-1px);
    }
    .info-card {
        background: #1f2937;
        padding: 1rem;
        border-radius: 8px;
        border-left: 5px solid #3b82f6;
        margin: 1rem 0;
        color: #f8fafc !important;
    }
    [data-testid="stSidebar"] {
        background-color: #1e1e1e;
        border-right: 1px solid #374151;
    }
    .sidebar-content {
        padding: 1rem;
        color: #f8fafc !important;
    }
    .footer {
        text-align: center;
        color: #9ca3af;
        margin-top: 2rem;
        font-size: 0.9rem;
    }
</style>
\"\"\", unsafe_allow_html=True)


# Configure Groq API
api_key = os.environ.get('GROQ_API_KEY')
if not api_key or api_key == "YOUR_GROQ_API_KEY_HERE":
    st.error("❌ Groq API Key not found! Get your FREE key at: https://console.groq.com")
    st.stop()

client = Groq(api_key=api_key)

# Initialize session state
if 'chat_history' not in st.session_state:
    st.session_state.chat_history = []

def call_groq_api(prompt):
    try:
        chat_completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile",  # Fast and powerful
            temperature=0.7,
            max_tokens=1024,
        )
        return chat_completion.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

def explain_concept(topic, difficulty_level, learning_style):
    prompt = f\"\"\"You are an expert teacher. Explain the following concept in simple, clear terms.

Topic: {topic}
Difficulty Level: {difficulty_level}
Learning Style: {learning_style}

Provide an explanation that:
1. Starts with a simple definition
2. Uses {learning_style} approach
3. Includes relevant examples
4. Is appropriate for {difficulty_level} level
5. Is easy to understand and engaging

Keep the explanation clear, concise, and student-friendly.\"\"\"
    return call_groq_api(prompt)

def summarize_notes(notes_text, summary_type):
    if summary_type == "Brief Summary":
        prompt = f"Summarize the following notes in 3-5 key bullet points:\\n\\n{notes_text}"
    elif summary_type == "Detailed Summary":
        prompt = f"Create a comprehensive summary of the following notes, organized by main topics:\\n\\n{notes_text}"
    else:
        prompt = f"Extract the most important key points and concepts from these notes:\\n\\n{notes_text}"
    return call_groq_api(prompt)

def generate_quiz(topic, num_questions, difficulty):
    prompt = f\"\"\"Generate {num_questions} multiple-choice questions about: {topic}

Difficulty: {difficulty}

Format each question as:
Q1: [Question text]
A) [Option A]
B) [Option B]
C) [Option C]
D) [Option D]
Correct Answer: [Letter]
Explanation: [Brief explanation]

Make the questions educational and thought-provoking.\"\"\"
    return call_groq_api(prompt)

def generate_flashcards(topic, num_cards):
    prompt = f\"\"\"Create {num_cards} flashcards for studying: {topic}

Format each flashcard as:
CARD [number]:
FRONT: [Question or term]
BACK: [Answer or definition]

Make them concise and memorable for effective studying.\"\"\"
    return call_groq_api(prompt)

def chat_with_ai(user_question):
    context = "\\n".join([f"Student: {q}\\nAI: {a}" for q, a in st.session_state.chat_history[-3:]])
    prompt = f\"\"\"You are a helpful study buddy assistant. Answer the student's question clearly and concisely.

Previous conversation:
{context}

Student's question: {user_question}

Provide a helpful, educational response that's easy to understand.\"\"\"
    return call_groq_api(prompt)

# Header
st.markdown('<div class="main-header"><h1>📚 AI-Powered Study Buddy</h1><p>Your personal AI assistant powered by Groq (Super Fast!)</p></div>', unsafe_allow_html=True)

# Sidebar
st.sidebar.header("🎓 Study Tools")
mode = st.sidebar.radio(
    "Select a tool:",
    ["💡 Explain Concept", "📝 Summarize Notes", "❓ Generate Quiz", "🎴 Create Flashcards", "💬 AI Chat Assistant"]
)

st.sidebar.markdown("---")
st.sidebar.markdown("### 📖 Quick Tips")
st.sidebar.info(\"\"\"
✅ Be specific with your questions
✅ Choose the right difficulty level
✅ Try different learning styles
✅ Practice with quizzes regularly
\"\"\")

# Main Content
if mode == "💡 Explain Concept":
    st.subheader("💡 Explain Any Concept")

    col1, col2 = st.columns([2, 1])

    with col1:
        topic = st.text_input(
            "What do you want to learn?",
            placeholder="e.g., Photosynthesis, Quantum Physics, Machine Learning, etc."
        )

        col_a, col_b = st.columns(2)
        with col_a:
            difficulty = st.selectbox("Difficulty Level:", ["Beginner", "Intermediate", "Advanced"])
        with col_b:
            learning_style = st.selectbox(
                "Learning Style:",
                ["Visual (with examples)", "Analytical (step-by-step)", "Practical (real-world)", "Simple (ELI5)"]
            )

    with col2:
        st.markdown('<div class="info-card"><strong>💡 How it works:</strong><br>Our AI will explain the concept in simple terms tailored to your learning style and level!</div>', unsafe_allow_html=True)

    if st.button("🚀 Explain This!"):
        if topic.strip():
            with st.spinner("🤔 AI is preparing your explanation..."):
                explanation = explain_concept(topic, difficulty, learning_style)
                st.success("✨ Explanation Generated!")
                st.markdown(explanation)
        else:
            st.warning("⚠️ Please enter a topic to explain.")

elif mode == "📝 Summarize Notes":
    st.subheader("📝 Summarize Your Study Notes")

    col1, col2 = st.columns([2, 1])

    with col1:
        notes = st.text_area(
            "Paste your notes here:",
            height=250,
            placeholder="Paste your lecture notes, study material, or any text you want summarized..."
        )
        summary_type = st.radio("Summary Type:", ["Brief Summary", "Detailed Summary", "Key Points Only"], horizontal=True)

    with col2:
        st.markdown('<div class="info-card"><strong>📝 Summary Types:</strong><br><br><b>Brief:</b> 3-5 key points<br><br><b>Detailed:</b> Comprehensive overview<br><br><b>Key Points:</b> Essential concepts only</div>', unsafe_allow_html=True)

    if st.button("✨ Summarize Now"):
        if notes.strip():
            with st.spinner("📖 AI is analyzing your notes..."):
                summary = summarize_notes(notes, summary_type)
                st.success("📋 Summary Generated!")
                st.markdown(summary)
        else:
            st.warning("⚠️ Please paste your notes first.")

elif mode == "❓ Generate Quiz":
    st.subheader("❓ Generate Practice Quiz")

    col1, col2 = st.columns([2, 1])

    with col1:
        quiz_topic = st.text_input("Quiz Topic:", placeholder="e.g., World War 2, Calculus, Python Programming")
        col_a, col_b = st.columns(2)
        with col_a:
            num_questions = st.slider("Number of Questions:", 3, 10, 5)
        with col_b:
            quiz_difficulty = st.selectbox("Difficulty:", ["Easy", "Medium", "Hard"])

    with col2:
        st.markdown('<div class="info-card"><strong>🎯 Quiz Benefits:</strong><br>• Test your knowledge<br>• Identify weak areas<br>• Reinforce learning<br>• Build confidence</div>', unsafe_allow_html=True)

    if st.button("🎲 Generate Quiz"):
        if quiz_topic.strip():
            with st.spinner("🎯 Creating your quiz..."):
                quiz = generate_quiz(quiz_topic, num_questions, quiz_difficulty)
                st.success("🎯 Quiz Generated!")
                st.markdown(quiz)
        else:
            st.warning("⚠️ Please enter a quiz topic.")

elif mode == "🎴 Create Flashcards":
    st.subheader("🎴 Generate Study Flashcards")

    col1, col2 = st.columns([2, 1])

    with col1:
        flashcard_topic = st.text_input("Flashcard Topic:", placeholder="e.g., Spanish Vocabulary, Chemistry Formulas, History Dates")
        num_cards = st.slider("Number of Flashcards:", 5, 20, 10)

    with col2:
        st.markdown('<div class="info-card"><strong>🎴 Flashcards are great for:</strong><br>• Quick revision<br>• Memorization<br>• Active recall<br>• Spaced repetition</div>', unsafe_allow_html=True)

    if st.button("✨ Create Flashcards"):
        if flashcard_topic.strip():
            with st.spinner("🎴 Generating flashcards..."):
                flashcards = generate_flashcards(flashcard_topic, num_cards)
                st.success("🎴 Flashcards Generated!")
                st.markdown(flashcards)
                st.info("💡 Tip: Review these flashcards daily for best results!")
        else:
            st.warning("⚠️ Please enter a topic for flashcards.")

elif mode == "💬 AI Chat Assistant":
    st.subheader("💬 Chat with Your AI Study Buddy")
    st.markdown('<div class="info-card">Ask me anything! I can help explain concepts, solve problems, clarify doubts, or discuss study topics.</div>', unsafe_allow_html=True)

    for question, answer in st.session_state.chat_history:
        with st.container():
            st.markdown(f"**👤 You:** {question}")
            st.info(f"**🤖 AI Study Buddy:** {answer}")

    user_question = st.text_input("Ask your question:", placeholder="e.g., How does gravity work? What is the difference between RNA and DNA?")

    col1, col2 = st.columns([1, 5])
    with col1:
        if st.button("💬 Ask"):
            if user_question.strip():
                with st.spinner("🤔 Thinking..."):
                    answer = chat_with_ai(user_question)
                    st.session_state.chat_history.append((user_question, answer))
                    st.rerun()
            else:
                st.warning("⚠️ Please enter a question.")
    with col2:
        if st.button("🗑️ Clear Chat"):
            st.session_state.chat_history = []
            st.rerun()

st.markdown("---")
st.markdown("<p style='text-align:center;color:#666;'>Powered by Groq AI ⚡ | Study Smart, Not Hard! 📚✨</p>", unsafe_allow_html=True)
"""

with open("temp_app.py", "w") as f:
    f.write(APP_CODE)

print("✅ App file written successfully.\n")

print("--- STEP 3 of 3: Launching the app... ---")

from pyngrok import ngrok
import time, sys

if not os.environ.get('GROQ_API_KEY') or os.environ.get('GROQ_API_KEY') == "YOUR_GROQ_API_KEY_HERE":
    print("❌ GROQ_API_KEY is not set properly. Get it at: https://console.groq.com")
    sys.exit()

if not os.environ.get('NGROK_AUTHTOKEN') or os.environ.get('NGROK_AUTHTOKEN') == "YOUR_NGROK_TOKEN_HERE":
    print("❌ NGROK_AUTHTOKEN is not set properly.")
    sys.exit()

ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])

get_ipython().system_raw('streamlit run temp_app.py --server.port 8501 --server.headless true &')
time.sleep(10)

try:
    public_url = ngrok.connect(8501)
    print("\n" + "="*60)
    print("🚀 YOUR AI STUDY BUDDY IS LIVE! 🚀")
    print(f"🔗 Public Link: {public_url}")
    print("="*60)
    print("\n📌 Click the link above to start studying!")
    print("⚠️ Keep this notebook running to keep the app alive.")
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    print("💡 Run Cell 2 first to close old tunnels, then run this cell again.")


--- Cleaning up old ngrok tunnels... ---
Found 1 active tunnels. Disconnecting...
  Closing: https://unchained-clinically-mariam.ngrok-free.dev


✅ All old tunnels closed!
✅ Streamlit processes killed!

--- STEP 1 of 3: Installing necessary packages... ---


✅ Packages installed successfully.

--- STEP 2 of 3: Defining and writing the Streamlit app code... ---
✅ App file written successfully.

--- STEP 3 of 3: Launching the app... ---

🚀 YOUR AI STUDY BUDDY IS LIVE! 🚀
🔗 Public Link: NgrokTunnel: "https://unchained-clinically-mariam.ngrok-free.dev" -> "http://localhost:8501"

📌 Click the link above to start studying!
⚠️ Keep this notebook running to keep the app alive.
